# Vayumukhi Dairy — free open-source LLM serverServes **Qwen2.5-VL-7B (AWQ)** from a free Colab T4 over an OpenAI-compatible API, so`@vmd/llm` can run Smart Scan, voice entry and the daily agent at **zero cost**.One model covers both roles: it is a strong document-OCR vision model *and* a capabletext model, and the AWQ 4-bit build fits comfortably in the T4's 16 GB.### Before you start1. **Runtime → Change runtime type → T4 GPU**. Without a GPU nothing here works.2. Run the cells top to bottom. **Cell 3 requires a runtime restart** — don't skip that note.3. Keep this tab open and visible while you develop.### Honest limitations- Google's [Colab FAQ](https://research.google.com/colaboratory/faq.html) prohibits  "bypassing the notebook UI to interact primarily via a web UI". This notebook is a  **local development and evaluation rig, not a production backend.** Expect the runtime  to be reclaimed.- Idle timeout ~90 min, hard cap ~12 h. **The tunnel URL changes every session** — you must  re-paste it into `.env.local` each time.- A free T4 needs **30–90 s per full-page scan** (vs ~3 s on hosted Claude). That is the  trade for $0. If a scan is slow, that's expected, not a bug.

## 1 · Confirm we actually have a GPU

In [ ]:
!nvidia-smi# T4 is Turing (SM75): no bfloat16, no FlashAttention-2. Both are handled by the# serve flags below. If this cell errors, you're on a CPU runtime — fix that first.

## 2 · Install vLLM⚠️ **vLLM pins its own PyTorch and will conflict with Colab's preinstalled one.**After this cell finishes you **must** restart the runtime(*Runtime → Restart session*), then continue from cell 3. Skipping the restart is thesingle most common reason this notebook fails.You do **not** need to re-run this cell after restarting — the install persists.

In [ ]:
!pip install -q vllmprint("\n\n=== NOW RESTART THE RUNTIME: Runtime -> Restart session, then run cell 3 ===")

## 3 · Configure (run this after the restart)

In [ ]:
import secrets, osMODEL = "Qwen/Qwen2.5-VL-7B-Instruct-AWQ"# The tunnel URL is public, so the server must require a token.LLM_API_KEY = secrets.token_urlsafe(32)os.environ["LLM_API_KEY"] = LLM_API_KEYprint("MODEL       :", MODEL)print("LLM_API_KEY :", LLM_API_KEY)

## 4 · Start the vLLM serverFlag notes — these are not arbitrary:- `--dtype half` — **required**. T4 is Turing and has no bf16 support.- `--quantization awq` — 4-bit weights (~5 GB), leaving room for the vision encoder + KV cache.- `--mm-processor-kwargs max_pixels` — caps vision tokens at ~1280. Without this a 12 MP phone  photo of a milk sheet blows past the context window and the request fails outright.- `--max-model-len 8192` — conservative, keeps KV cache within 16 GB.First run downloads ~6 GB of weights, so expect 3–6 minutes. Subsequent restarts are faster.

In [ ]:
import subprocess, time, os, itertoolsLOG = "/content/vllm.log"cmd = [    "vllm", "serve", MODEL,    "--quantization", "awq",    "--dtype", "half",    "--max-model-len", "8192",    "--gpu-memory-utilization", "0.90",    "--limit-mm-per-prompt", "image=1",    "--mm-processor-kwargs", '{"max_pixels": 1003520}',    "--api-key", os.environ["LLM_API_KEY"],    "--port", "8000",]with open(LOG, "wb") as f:    server = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)print("Booting vLLM (downloading weights on first run)...")for i in itertools.count():    time.sleep(5)    log = open(LOG, errors="ignore").read()    if "Application startup complete" in log:        print(f"\n✅ vLLM is up after ~{i*5}s")        break    if server.poll() is not None:        print("\n❌ vLLM died. Last 40 lines:\n")        print("\n".join(log.splitlines()[-40:]))        break    if i % 6 == 0:        tail = [l for l in log.splitlines() if l.strip()][-1:]        print(f"  {i*5}s… {tail[0][:110] if tail else ''}")

## 5 · Expose it with a Cloudflare quick tunnel`cloudflared` gives a public HTTPS URL with **no account and no signup**. The URL is randomand changes every time you run this.

In [ ]:
import subprocess, time, re!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64!chmod +x /content/cloudflaredTLOG = "/content/tunnel.log"with open(TLOG, "wb") as f:    tunnel = subprocess.Popen(        ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],        stdout=f, stderr=subprocess.STDOUT,    )PUBLIC_URL = Nonefor _ in range(40):    time.sleep(2)    m = re.search(r"https://[-\w]+\.trycloudflare\.com", open(TLOG, errors="ignore").read())    if m:        PUBLIC_URL = m.group(0)        breakprint("✅ " + PUBLIC_URL if PUBLIC_URL else "❌ No tunnel URL — check /content/tunnel.log")

## 6 · Paste this into `.env.local` (repo root)

In [ ]:
print(f'''LLM_PROVIDER="openai-compat"LLM_BASE_URL="{PUBLIC_URL}/v1"LLM_API_KEY="{os.environ["LLM_API_KEY"]}"LLM_MODEL_AGENT="{MODEL}"LLM_MODEL_FAST="{MODEL}"LLM_TIMEOUT_MS="120000"''')print("Then restart `pnpm dev` so Next.js picks up the new env.")

## 7 · Smoke test — text extractionMirrors what `extractMilkFromText` sends, including the grammar-constrained`response_format`. If this returns valid JSON, the voice/assistant path will work.

In [ ]:
import requests, jsonBASE = f"{PUBLIC_URL}/v1"HEAD = {"Authorization": f"Bearer {os.environ['LLM_API_KEY']}"}MILK_SCHEMA = {    "type": "object",    "properties": {        "litres":     {"type": ["number", "null"]},        "fatPct":     {"type": ["number", "null"], "minimum": 0, "maximum": 15},        "animalTag":  {"type": ["string", "null"]},        "animalName": {"type": ["string", "null"]},        "shift":      {"enum": ["morning", "evening", None]},        "confidence": {"type": "number", "minimum": 0, "maximum": 1},    },    "required": ["litres", "fatPct", "animalTag", "animalName", "shift", "confidence"],}r = requests.post(f"{BASE}/chat/completions", headers=HEAD, timeout=180, json={    "model": MODEL,    "max_tokens": 512,    "temperature": 0,    "messages": [{"role": "user", "content":        'Extract the milk session fields. Entry: "12.5 litres morning, fat 4.2%"'}],    "response_format": {"type": "json_schema",                        "json_schema": {"name": "milk_extraction", "schema": MILK_SCHEMA}},})r.raise_for_status()out = json.loads(r.json()["choices"][0]["message"]["content"])print(json.dumps(out, indent=2))assert out["litres"] == 12.5 and out["shift"] == "morning", "unexpected extraction"print("\n✅ text extraction OK")

## 8 · Smoke test — vision (Smart Scan)Generates a synthetic milk sheet and runs the real `SCAN_INSTRUCTION` against it, so you canverify the vision path without leaving the notebook. **This is the slow one — 30–90 s.**

In [ ]:
from PIL import Image, ImageDrawimport base64, io, time# Synthetic milk sheet standing in for a phone photo.img = Image.new("RGB", (900, 640), "white")d = ImageDraw.Draw(img)d.text((40, 30), "MILK SHEET - 12 Mar", fill="black")d.text((40, 90), "Animal    Litres   Fat%", fill="black")for i, (a, l, f) in enumerate([("Ganga", "12.5", "4.2"),                               ("Yamuna", "9.0", "3.8"),                               ("Kaveri", "11.2", "4.0")]):    d.text((40, 140 + i * 50), f"{a:<10}{l:<9}{f}", fill="black")buf = io.BytesIO(); img.save(buf, format="JPEG")b64 = base64.b64encode(buf.getvalue()).decode()display(img)SCAN_INSTRUCTION = (    "You read photos of dairy-farm paperwork. First CLASSIFY the document as one of: "    "milk_sheet (a daily milk log with rows of animals + litres), feed_sheet (a feed/fodder log), "    "expense (a bill/receipt/invoice), or other. Then EXTRACT and respond with ONLY a JSON object, "    "no prose. Include every readable row. Use null for anything illegible.")SCAN_SCHEMA = {    "type": "object",    "properties": {        "type":       {"enum": ["milk_sheet", "feed_sheet", "expense", "other"]},        "confidence": {"type": "number", "minimum": 0, "maximum": 1},        "rows": {"type": "array", "items": {"type": "object", "properties": {            "animal": {"type": ["string", "null"]},            "litres": {"type": ["number", "null"]},            "fatPct": {"type": ["number", "null"]},            "shift":  {"enum": ["morning", "evening", None]},        }}},        "title": {"type": ["string", "null"]},    },    "required": ["type", "confidence"],}t0 = time.time()r = requests.post(f"{BASE}/chat/completions", headers=HEAD, timeout=300, json={    "model": MODEL,    "max_tokens": 1500,    "temperature": 0,    "messages": [{"role": "user", "content": [        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},        {"type": "text", "text": SCAN_INSTRUCTION},    ]}],    "response_format": {"type": "json_schema",                        "json_schema": {"name": "scan_result", "schema": SCAN_SCHEMA}},})r.raise_for_status()out = json.loads(r.json()["choices"][0]["message"]["content"])print(json.dumps(out, indent=2))print(f"\n⏱  {time.time() - t0:.1f}s   (30-90s is normal on a free T4)")assert out["type"] == "milk_sheet", f"misclassified as {out['type']}"print("✅ vision scan OK — the app is ready to use this server")

## Troubleshooting| Symptom | Cause / fix ||---|---|| `ImportError` / torch version errors in cell 4 | You skipped the **runtime restart** after cell 2. Restart, then resume at cell 3. || `CUDA out of memory` | Lower `--gpu-memory-utilization` to `0.85`, or `--max-model-len` to `4096`. || `bfloat16 is only supported on GPUs with compute capability of at least 8.0` | You dropped `--dtype half`. T4 is Turing; put it back. || `No supported backend` / attention errors | You changed the model to a **Qwen3-VL** build. Those have no Turing backend in vLLM ([vllm#29743](https://github.com/vllm-project/vllm/issues/29743)). Stay on Qwen2.5-VL. || Request hangs, then the app logs a timeout | Normal for a big image. Raise `LLM_TIMEOUT_MS`, or downscale before upload. || Tunnel URL stops working | The runtime was reclaimed. Re-run cells 3–6 and re-paste `.env.local`. || App silently returns `type: "other"` scans | `.env.local` isn't loaded, or `LLM_BASE_URL` lacks the trailing `/v1`. |**To stop everything:** *Runtime → Disconnect and delete runtime*.